# 08 ヒープとハッシュ: Heaps and Hashes

- 最小または最大の要素を効率的に取得するデータ構造としてヒープがある。ヒープは完全二分木として実装できる。最小(最大)要素の取り出しや、要素の追加を高速に実行できる。
- データの集合を効率的に管理するためにハッシュテーブルが使われる。ハッシュ関数を用いてデータをインデックス化し、平均してO(1)の時間で要素の挿入、削除、検索が可能となる。

- Heaps are data structures that allow efficient retrieval of the minimum or maximum element. They can be implemented as complete binary trees, enabling fast extraction of the min (or max) element and addition of new elements.
- Hash tables are used to efficiently manage collections of data. By using hash functions to index data, they allow for average O(1) time complexity for insertion, deletion, and search operations.
----

## ヒープ (Heaps)
- ここでは、最小値を効率的に取得することにする
- 各ノードの値は、その子ノードの値以下となる
- 要素は、リストとして保持できる
----
- Here, we focus on efficiently retrieving the minimum value.
- The value of each node is less than or equal to the values of its child nodes.
- Elements can be stored as a list.

In [ ]:
import random

In [ ]:
class BinaryHeap:
    """
    最小ヒープの簡単な実装（リストを使用）
    A simple implementation of a min-heap using a list.
    """
    def __init__(self):
        self._data = [None]  # 1-based index; index 0 is unused
        self._size = 0
    
    def min_heapify(self, index):
        smallest = index
        left = 2 * index
        right = 2 * index + 1
        
        if left <= self._size and self._data[left] < self._data[smallest]:
            smallest = left
        if right <= self._size and self._data[right] < self._data[smallest]:
            smallest = right
        
        if smallest != index:
            self._data[index], self._data[smallest] = self._data[smallest], self._data[index]
            self.min_heapify(smallest)
    
    def build_min_heap(self, array):
        self._data = [None] + array[:]  # 1-based index
        self._size = len(array)
        for i in range(self._size // 2, 0, -1):
            self.min_heapify(i)

    def insert(self, value):
        self._data.append(value)
        self._size += 1
        self._shift_up(self._size)
    def _shift_up(self, index):
        parent = index // 2
        if parent > 0 and self._data[index] < self._data[parent]:
            self._data[index], self._data[parent] = self._data[parent], self._data[index]
            self._shift_up(parent)
    
    def pop_min(self):
        if self.is_empty:
            raise IndexError("Heap is empty")
        min_value = self._data[1]
        self._data[1] = self._data[self._size]
        self._data.pop()  # Remove the last element
        self._size -= 1
        if not self.is_empty:
            self.min_heapify(1)
        return min_value
    @property
    def is_empty(self):
        return self._size == 0
    @property
    def heap_list(self):
        return self._data[1:]  # Exclude the first None element

In [ ]:
data = [4,1,7,9,4,2,3,8,5,6]

print("Input data:", data)
heap = BinaryHeap()
heap.build_min_heap(data)
print(heap.heap_list)
print('add new value 3 to heap')
heap.insert(3)
print(heap.heap_list)
print('pop min value from heap')
v = heap.pop_min()
print(f"Popped min value: {v}")
print(heap.heap_list)
# if not heap.is_empty:
#     v = heap.pop_min()
#     print(f"Popping values from heap: {v}")
# print(heap.heap_list)

## ハッシュテーブル (Hash Tables)
- ハッシュ関数を用いて、データをインデックス化する
- 簡単のために、値の範囲を制限してリストに保存する

- By using hash functions to index data.
- For simplicity, we limit the range of values and store them in a list.
----

### ハッシュ関数: Hash Functions
- ハッシュ関数は、データを固定サイズの値に変換する関数
- 不変なデータを入力として、同じハッシュ値を生成する

- A hash function is a function that converts data into a fixed-size value.
- It generates the same hash value for the same input data.


In [ ]:
a = 29873948
print(hash(a))
b = "Hello, World!"
print(hash(b))
c = (1, 2, 3)
print(hash(c))

大量のデータを扱うための高度なhash関数が用意されている。
以下では、256ビットのハッシュ値を生成するSHA-256を使用する例を示す。

There are advanced hash functions available for handling large amounts of data.
The following example demonstrates the use of SHA-256, which generates a 256-bit hash value.

In [ ]:
import hashlib
m = hashlib.sha256()
m.update(b"Hello, World!")
print(m.hexdigest())

m = hashlib.sha256()
d = (1,2,3)
m.update(str(d).encode())
print(m.hexdigest())

In [ ]:
class SimpleHashTable:
    """
    A simple hash table implementation using a list.
    """
    def __init__(self, size):
        self.size = size
        self.table:list = [None] * size # initialize the table with None

    def _hash(self, key):# simple hash function
        _,b = divmod(hash(key), self.size)
        return b

    def insert(self, value):
        index = self._hash(value)
        if self.table[index] is not None:
            print(f"Collision occurred for key: {value} at index: {index}")
        self.table[index] = value

    def get(self, value) :
        index = self._hash(value)
        return index,self.table[index]

In [ ]:
max_value = 64
data = ["orange", "apple", "banana", "grape", "kiwi", "mango", "peach", "plum", 
        "pear", "cherry","blueberry","strawberry","raspberry","blackberry","watermelon","melon"]
for v in data:
    print(f"Hash for {v}: {hash(v)}")
hash_table = SimpleHashTable(size=max_value)
for v in data:
    hash_table.insert(v)
print("--------------------")
print("retrieving values from hash table:")
for v in data:
    index,value = hash_table.get(v)
    print(f"Value: {v} at index: {index}")

単純に、入力データを文字列として扱い、その文字列のハッシュ値を生成し、上限値の剰余をインデクスとすると、要素の可能性が上限値を超える場合に、インデクスが衝突する可能性がある。そこで、各インデクスの下にリストを用意し、衝突した要素をそのリストに追加する方法により、衝突を解決することができる。

Simply treating the input data as a string, generating its hash value, and using the remainder of the hash value divided by an upper limit as an index can lead to collisions when the number of possible elements exceeds the upper limit. To resolve collisions, we can prepare a list under each index and add colliding elements to that list.

In [ ]:
class SimpleHashTable2(SimpleHashTable):
    """
        衝突を避けるために、各インデクスにリストを使用する方法を用いて、集合(set)の実装

        A simple implementation of a set using a hash table with separate chaining to handle collisions.
    """
    def __init__(self, size=1000):
        super().__init__(size)
        for i in range(size):
            self.table[i] = [] # initialize each index with an empty list

    def insert(self, value):
        index = self._hash(value)
        if value not in self.table[index]:
            self.table[index].append(value)

    def get(self, key):
        index,o = super().get(key)
        if o is not None:
            if key in o:
                return index,key
        return -1,None



max_value = 64
data = ["orange", "apple", "banana", "grape", "kiwi", "mango", "peach", "plum", 
        "pear", "cherry","blueberry","strawberry","raspberry","blackberry","watermelon","melon"]
for v in data:
    print(f"Hash for {v}: {hash(v)}")
hash_table = SimpleHashTable2(size=max_value)
for v in data:
    hash_table.insert(v)
print("--------------------")
print("retrieving values from hash table:")
for v in data:
    index, value = hash_table.get(v)
    print(f"Value: {v} at index: {index}")